# Event Study Visualisation and Pre-Trend Testing

**Module 04 | Estimated time: 15 minutes**

## Learning Objectives

By the end of this notebook, you will be able to:
1. Build publication-quality event study plots from panel data
2. Implement the event study regression with proper normalisation
3. Run and interpret pre-trend tests (Wald/F-test)
4. Identify patterns that indicate parallel trends violations
5. Build confidence bands around event study estimates

## Background

Event study plots have become the standard visual for DiD analyses. They answer two questions simultaneously:
1. **Before treatment:** Did treated and control units have parallel trends? (Pre-period $\beta_k \approx 0$)
2. **After treatment:** When did the effect emerge and how did it evolve? (Post-period $\beta_k$ trajectory)

This notebook teaches you to build these plots from scratch, understand their construction, and use them for diagnostic testing.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy.stats import chi2
import warnings
warnings.filterwarnings('ignore')

np.random.seed(314)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
print("Libraries loaded.")

## 1. Generate Panel Data with Known Treatment Effects

We'll work with a single treatment cohort (all units treated at the same time) to build the event study concept cleanly, then extend to staggered adoption.

True treatment effect:
- k < 0: no effect (pre-period)
- k = 0: immediate jump of 2.0
- k > 0: growing effect (+0.5 per period), capping at k=4

In [ ]:
# Panel with single treatment period: T* = 8
N_UNITS = 100
N_PERIODS = 16
TREAT_PERIOD = 8
N_TREATED = 50

# True effect profile by event time
def true_effect(k):
    """True treatment effect at event time k."""
    if k < 0:
        return 0.0
    elif k == 0:
        return 2.0
    else:
        return 2.0 + min(k, 4) * 0.5  # grows for 4 periods, then stabilises

rows = []
for u in range(N_UNITS):
    unit_fe = np.random.normal(0, 1.5)
    treated = 1 if u < N_TREATED else 0
    for t in range(1, N_PERIODS + 1):
        k = t - TREAT_PERIOD  # event time
        tau = true_effect(k) if treated else 0
        y = 3.0 + 0.3 * t + unit_fe + tau + np.random.normal(0, 1.0)
        rows.append({
            'unit': u, 'period': t, 'treated': treated,
            'event_time': k, 'true_tau': tau, 'outcome': y
        })

df = pd.DataFrame(rows)

# Print true effects
print("True treatment effect profile:")
for k in range(-4, 9):
    print(f"  k={k:>3}: τ = {true_effect(k):.1f}")

## 2. The Event Study Regression

The event study regression estimates a separate coefficient for each period relative to treatment:

$$Y_{it} = \alpha_i + \lambda_t + \sum_{k \neq -1} \beta_k \cdot \mathbf{1}[t - T^* = k] \cdot D_i + \epsilon_{it}$$

Where $k = t - T^*$ is the event time. The period $k = -1$ is omitted (normalised to zero) as the baseline.

In [ ]:
# Create event-time interaction dummies
# Event times to include: -4 to 8 (excluding -1 baseline)
event_times = [k for k in range(-4, 9) if k != -1]

for k in event_times:
    col_name = f'D_k{k:+d}' if k != 0 else 'D_k0'
    df[col_name] = ((df['event_time'] == k) & (df['treated'] == 1)).astype(float)

# Build the formula
interaction_terms = ' + '.join([f'D_k{k:+d}' if k != 0 else 'D_k0' for k in event_times])
formula = f'outcome ~ {interaction_terms} + C(unit) + C(period)'

# Fit the model
model = smf.ols(formula, data=df).fit(cov_type='HC1')

# Extract event study coefficients and SEs
es_coefs = []
for k in event_times:
    col_name = f'D_k{k:+d}' if k != 0 else 'D_k0'
    coef = model.params[col_name]
    se = model.bse[col_name]
    ci = model.conf_int().loc[col_name].values
    es_coefs.append({
        'k': k, 'beta': coef, 'se': se,
        'ci_lo': ci[0], 'ci_hi': ci[1],
        'true_tau': true_effect(k)
    })

# Add the normalised baseline k=-1
es_coefs.append({'k': -1, 'beta': 0, 'se': 0,
                  'ci_lo': 0, 'ci_hi': 0, 'true_tau': 0})

es_df = pd.DataFrame(es_coefs).sort_values('k').reset_index(drop=True)
print(f"Event study estimated for {len(es_df)} periods")
print(es_df[['k', 'beta', 'se', 'true_tau']].round(3))

## 3. Publication-Quality Event Study Plot

The standard event study visualisation with confidence bands, zero reference line, treatment onset marker, and comparison to the true effect.

In [ ]:
def plot_event_study(es_df, title='Event Study', show_true=True, ax=None):
    """Publication-quality event study plot."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(11, 5))
    else:
        fig = ax.figure
    
    pre = es_df[es_df['k'] < 0]
    post = es_df[es_df['k'] >= 0]
    
    # Pre-period: blue dots with CIs
    ax.errorbar(pre['k'], pre['beta'],
                yerr=[pre['beta'] - pre['ci_lo'], pre['ci_hi'] - pre['beta']],
                fmt='o', color='steelblue', capsize=4, linewidth=1.5, markersize=7,
                label='Pre-treatment estimates')
    
    # Post-period: orange dots with CIs
    ax.errorbar(post['k'], post['beta'],
                yerr=[post['beta'] - post['ci_lo'], post['ci_hi'] - post['beta']],
                fmt='s', color='darkorange', capsize=4, linewidth=1.5, markersize=7,
                label='Post-treatment estimates')
    
    # True effect line
    if show_true and 'true_tau' in es_df.columns:
        ax.plot(es_df['k'], es_df['true_tau'], '--', color='green',
                linewidth=2, alpha=0.7, label='True effect')
    
    # Reference lines
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
    ax.axvline(-0.5, color='red', linewidth=1.5, linestyle=':', alpha=0.7)
    ax.axvspan(-5, -0.5, alpha=0.03, color='blue', label='Pre-period')
    ax.axvspan(-0.5, 9, alpha=0.03, color='orange', label='Post-period')
    
    # Baseline marker
    ax.scatter([-1], [0], color='black', s=100, zorder=10, marker='D',
               label='Baseline (k=-1, normalised to 0)')
    
    ax.set_xlabel('Event Time k (periods relative to treatment)')
    ax.set_ylabel('Treatment Effect Estimate (β_k)')
    ax.set_title(title)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(axis='y', alpha=0.3)
    
    return fig, ax

fig, ax = plot_event_study(
    es_df,
    title='Event Study: Job Training Program\n(Pre-period: test of parallel trends; Post-period: treatment dynamics)'
)
plt.tight_layout()
plt.savefig('../resources/event_study.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Pre-Trend Test: Wald/F-test

Formally test whether the pre-period coefficients are jointly equal to zero.

In [ ]:
def pre_trend_wald_test(model, pre_event_times):
    """
    Wald test for joint significance of pre-period event study coefficients.
    H0: β_k = 0 for all k < -1 (excluding baseline k=-1)
    """
    # Get variable names for pre-period dummies
    pre_vars = []
    for k in pre_event_times:
        if k == -1:  # baseline, excluded
            continue
        col_name = f'D_k{k:+d}'
        if col_name in model.params.index:
            pre_vars.append(col_name)
    
    if not pre_vars:
        print("No pre-period variables found.")
        return
    
    # Extract coefficients and covariance
    pre_coefs = model.params[pre_vars].values
    pre_cov = model.cov_HC1[np.ix_(
        [list(model.params.index).index(v) for v in pre_vars],
        [list(model.params.index).index(v) for v in pre_vars]
    )]
    
    # Wald statistic
    W = pre_coefs @ np.linalg.solve(pre_cov, pre_coefs)
    df_test = len(pre_vars)
    p_value = 1 - chi2.cdf(W, df=df_test)
    
    print(f"Pre-Trend Wald Test")
    print(f"H0: β_k = 0 for all k ∈ {pre_event_times} (excluding k=-1 baseline)")
    print(f"Variables tested: {pre_vars}")
    print(f"Wald statistic: W = {W:.3f}")
    print(f"Degrees of freedom: {df_test}")
    print(f"p-value: {p_value:.4f}")
    if p_value > 0.05:
        print("Result: Fail to reject H0 — pre-trends are consistent with zero")
        print("        (Supports, but does not prove, parallel trends assumption)")
    else:
        print("Result: REJECT H0 — significant pre-existing trends detected")
        print("        (Parallel trends assumption is questionable)")
    return W, p_value

pre_times = list(range(-4, 0))
W, p = pre_trend_wald_test(model, pre_times)

## 5. What a Parallel Trends Violation Looks Like

Let's simulate a pre-existing upward trend in the treated group and see how it appears in the event study plot.

In [ ]:
# Add a pre-existing trend to treated units
df_biased = df.copy()
PRE_TREND_SLOPE = 0.3  # treated units growing 0.3 units per period faster before treatment

df_biased['outcome_biased'] = df_biased['outcome'] + (
    df_biased['treated'] * PRE_TREND_SLOPE * df_biased['period']
)

# Re-estimate event study on biased data
formula_biased = f'outcome_biased ~ {interaction_terms} + C(unit) + C(period)'
model_biased = smf.ols(formula_biased, data=df_biased).fit(cov_type='HC1')

es_biased = []
for k in event_times:
    col_name = f'D_k{k:+d}' if k != 0 else 'D_k0'
    coef = model_biased.params[col_name]
    se = model_biased.bse[col_name]
    ci = model_biased.conf_int().loc[col_name].values
    es_biased.append({'k': k, 'beta': coef, 'se': se,
                       'ci_lo': ci[0], 'ci_hi': ci[1], 'true_tau': true_effect(k)})
es_biased.append({'k': -1, 'beta': 0, 'se': 0, 'ci_lo': 0, 'ci_hi': 0, 'true_tau': 0})
es_biased_df = pd.DataFrame(es_biased).sort_values('k').reset_index(drop=True)

# Side-by-side: good vs biased
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

plot_event_study(es_df, title='Clean: No Pre-Trend\n(Parallel trends holds)', ax=axes[0])
plot_event_study(es_biased_df, title='VIOLATION: Pre-Existing Trend\n(Treated units growing faster before treatment)', ax=axes[1])

# Highlight pre-period on right panel
axes[1].axhspan(-1, 5, xmin=0, xmax=0.35, alpha=0.08, color='red')
axes[1].text(-3.5, 4.5, 'Pre-trend!', color='red', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("Pre-trend test on biased data:")
pre_trend_wald_test(model_biased, pre_times)

## 6. Confidence Bands and Uncertainty

Event studies should always show uncertainty. We'll add simultaneous confidence bands using a Bonferroni correction, which is more conservative but accounts for multiple comparisons.

In [ ]:
from scipy.stats import norm

# Simultaneous confidence bands (Bonferroni corrected)
n_comparisons = len(event_times)
alpha = 0.05
bonferroni_z = norm.ppf(1 - alpha / (2 * n_comparisons))
standard_z = norm.ppf(0.975)  # 1.96 for 95% CI

print(f"Standard 95% CI uses z = {standard_z:.2f}")
print(f"Bonferroni-corrected CI uses z = {bonferroni_z:.2f} (more conservative)")

fig, ax = plt.subplots(figsize=(11, 5))

# Pointwise CIs (narrower)
ax.fill_between(es_df['k'], es_df['beta'] - standard_z * es_df['se'],
                es_df['beta'] + standard_z * es_df['se'],
                alpha=0.2, color='steelblue', label='Pointwise 95% CI')

# Simultaneous bands (wider)
ax.fill_between(es_df['k'], es_df['beta'] - bonferroni_z * es_df['se'],
                es_df['beta'] + bonferroni_z * es_df['se'],
                alpha=0.1, color='steelblue', label='Simultaneous 95% CI (Bonferroni)')

# Point estimates
ax.plot(es_df['k'], es_df['beta'], 'o-', color='steelblue', linewidth=2, markersize=6,
        label='Event study estimate')
ax.plot(es_df['k'], es_df['true_tau'], '--', color='green', linewidth=2, alpha=0.7,
        label='True effect')

ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax.axvline(-0.5, color='red', linestyle=':', linewidth=1.5, alpha=0.7, label='Treatment')
ax.scatter([-1], [0], color='black', s=100, zorder=10, marker='D', label='Baseline')

ax.set_xlabel('Event Time k')
ax.set_ylabel('β_k')
ax.set_title('Event Study with Pointwise and Simultaneous Confidence Bands')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Interpreting the Full Event Study

Let's produce a final annotated interpretation of our main event study.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

# Shaded CI
ax.fill_between(es_df['k'],
                es_df['beta'] - 1.96 * es_df['se'],
                es_df['beta'] + 1.96 * es_df['se'],
                alpha=0.2, color='steelblue')

# Point estimates
ax.plot(es_df['k'], es_df['beta'], 'o-', color='steelblue',
        linewidth=2, markersize=7, label='Event study estimate (95% CI)')
ax.plot(es_df['k'], es_df['true_tau'], '--', color='green',
        linewidth=2, alpha=0.6, label='True effect')

# Reference
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(-0.5, color='red', linewidth=2, linestyle='--', label='Treatment onset')
ax.scatter([-1], [0], color='black', s=120, zorder=10, marker='D')

# Annotations
ax.annotate('Pre-period: β_k ≈ 0\n→ Supports parallel trends',
            xy=(-3, 0.2), fontsize=10, color='steelblue',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.4))

ax.annotate('Immediate effect\nat k=0',
            xy=(0, es_df[es_df['k']==0]['beta'].values[0]),
            xytext=(1.5, 2.5),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10)

ax.annotate('Effect grows\nover time',
            xy=(4, es_df[es_df['k']==4]['beta'].values[0]),
            xytext=(5.5, 3.0),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=10)

ax.set_xlabel('Event Time k (periods relative to treatment)', fontsize=12)
ax.set_ylabel('Estimated Treatment Effect (β_k)', fontsize=12)
ax.set_title('Annotated Event Study: Reading the Plot', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\nInterpretation summary:")
print(f"  Pre-period (k=-4 to -2): β range = "
      f"[{es_df[es_df['k']<-1]['beta'].min():.2f}, {es_df[es_df['k']<-1]['beta'].max():.2f}]")
print(f"  All pre-period CIs include zero: {(es_df[es_df['k']<-1]['ci_lo'] < 0).all() and (es_df[es_df['k']<-1]['ci_hi'] > 0).all()}")
print(f"  Post-period range: β = "
      f"[{es_df[es_df['k']>=0]['beta'].min():.2f}, {es_df[es_df['k']>=0]['beta'].max():.2f}]")

## Self-Check

1. Change `TREAT_PERIOD` from 8 to 12 and rerun. How does this affect the number of pre-period versus post-period estimates? What are the tradeoffs of having few pre-periods for parallel trends testing?

2. Set `PRE_TREND_SLOPE = 0.6` and observe how the pre-trend test performs. At what slope does the test reliably detect the violation with p < 0.05?

3. Modify `true_effect()` so the effect fades out to zero by k=6. Plot the event study. Can you distinguish between a fading effect and a parallel trends violation based on the plot alone?

4. The baseline period is k=-1. Try using k=0 as the baseline instead (drop the k=0 dummy from the regression). How do the estimated coefficients change in interpretation?

---
**Previous:** `02_staggered_did.ipynb`
**Next:** Module 05 — `01_sharp_rdd.ipynb`